# S4 pilot — local generation on Kaggle T4

**This notebook is a RUNNER only.** It clones the repo, installs, and calls scripts.
No logic lives here — `CLAUDE.md`: *"If logic lives in a notebook cell, it cannot
enter the paper."*

⛔ **The pilot is NOT A RESULT.** It selects a generator; it measures nothing.

## The two arms

| arm | model | difference |
|---|---|---|
| A | `google/gemma-3-1b-it` | general multilingual |
| B | `md-nishat-008/TigerLLM-1B-it` | **same base, Bangla-adapted** |

Verified from `config.json`: both are `Gemma3ForCausalLM`, hidden 1152, 26 layers,
vocab 262144. **One variable: Bangla adaptation.**

⚠️ The TigerLLM paper (arXiv 2503.10995) says its 1B is built on **LLaMA-3.2**.
The uploaded weights are **Gemma-3**. The paper's benchmark table therefore does
not describe these weights, and no claim here rests on it.

**Settings: GPU T4 x2 · Internet ON · attach `bn_clean.csv` as a Dataset.**


## 0. Smoke test FIRST — before anything else

Gemma-3 ships **bf16** weights and a T4 (Turing, sm_75) has no bf16. Casting to
fp16 is a known risk for this family. So the first thing this notebook does is
generate one sample and **print it** — two of the three bugs found on 2026-08-11
were caught by reading a rendered artifact, not by a test.

**If the output is empty, English, or garbage, stop here.**


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate 2>&1 | tail -2


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'md-nishat-008/TigerLLM-1B-it'   # arm B; swap for arm A to compare
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map='auto')

prompt = 'তুমি একজন সাধারণ বাংলাদেশি দর্শক। একটি সিনেমা নিয়ে ছোট একটি মন্তব্য লেখো।'
text = tok.apply_chat_template([{'role':'user','content':prompt}],
                               tokenize=False, add_generation_prompt=True)
enc = tok(text, return_tensors='pt', add_special_tokens=False).to(m.device)
t0 = time.time()
out = m.generate(**enc, do_sample=True, temperature=0.8, top_p=0.9, max_new_tokens=200)
print(f'{time.time()-t0:.1f}s')
print(tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True))


### Read the output above before continuing.

- **Bangla, coherent, comment-like** → continue.
- **Empty / `nan` / repeated tokens** → fp16 overflow. Try `bfloat16` (will fail on
  T4) or fall back to the API path. Record the failure; do not work around it.
- **English** → the chat template or the model is not doing what we assume.


## 1. Repo, data, index


In [ ]:
!git clone -q https://github.com/alphapie77/BSc_Thesis.git repo
%cd repo
!pip -q install chromadb sentence-transformers pyyaml 2>&1 | tail -2


In [ ]:
# bn_clean.csv is gitignored (size + licence), so it arrives as a Kaggle Dataset.
!mkdir -p data/cleaned
!cp /kaggle/input/bn-clean/bn_clean.csv data/cleaned/bn_clean.csv
!ls -l data/cleaned/


In [ ]:
# Rebuild the R1-only index here. It refuses to run if an R2 or Gold-300 id
# reaches it (inviolable rules 4 and 5), checked twice by two mechanisms.
!python src/agents/build_index.py --config configs/s4_index.yaml


**Expect `886` rows and a digest starting `85fc2d7d`.** A different digest means
different rows went in, and nothing downstream is comparable to the local run.


## 2. Dry run — print the real prompt

No generation. Reads the actual retrieval and prints the full prompt for both
levels and both language arms.


In [ ]:
!python src/agents/run_pilot.py --dry-run 2>&1 | head -80


## 3. The pilot

Resumable: every generation is appended to JSONL as it completes, and a re-run
skips what is on disk. A 12-hour session cap cannot lose the run.


In [ ]:
!python src/agents/run_pilot.py --config configs/s4_pilot_local.yaml


## 4. Save the outputs back

⚠️ **Kaggle resets disk between sessions.** The JSONL is the reproducibility
artifact — `2601.17768` means these generations cannot be regenerated — so it must
leave the notebook. Download `/kaggle/working/` and commit it to the repo.


In [ ]:
!cp results/pilot_s4_generations.jsonl /kaggle/working/ 2>/dev/null
!cp results/pilot_s4_model_choice.* /kaggle/working/ 2>/dev/null
!python src/common/env_snapshot.py && cp results/env_snapshot.json /kaggle/working/env_snapshot_s4_kaggle.json
!ls -lh /kaggle/working/
